In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# important workflow libraries:
import torch, gc, os, sys, tiktoken, json, types
from torch.utils.data import Dataset
from transformers import PretrainedConfig
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer
import random

In [ ]:
# storing paths:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [ ]:
# adding paths to system:
sys.path.append(proj_path)
sys.path.append(data_path)
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [ ]:
# getting GPU if available:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
# importing modules:
from gpt_model import GPTModel
from config import CONFIG
model = GPTModel(CONFIG)

In [ ]:
torch.cuda.empty_cache()
gc.collect()

60

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/llm_from_scratch/pretrained_weights.pth"))
print("Weights_loaded successfully.")

Weights_loaded successfully.


In [ ]:
# to load jsonl and convert data into required formate:
def load_and_convert_data(input_file):
    data = []
    with open(input_file, 'r') as f:
        for line in f:
            item = json.loads(line)
            if "conversations" in item and len(item["conversations"]) >= 2:
                data.append({
                    "messages": [
                        {"role": "user", "content": item["conversations"][0]},
                        {"role": "assistant", "content": item["conversations"][1]}
                    ]
                })
    return data

In [ ]:
instruction_data = load_and_convert_data("/content/drive/MyDrive/llm_from_scratch/datasets/instruction_data_lima.jsonl")

In [ ]:
print(len(instruction_data),  "\n")
instruction_data[44]

1030 



{'messages': [{'role': 'user',
   'content': "Is there a Git for data? The key improvement I'd want is to Diff/Merge more intelligently. e.g. in CSV rather than line vs line comparison, it would do cell vs cell.\nAnd ordering is usually not significant, e.g. rows in a CSV, whereas Git does care and presents the user with 'conflicts'."},
  {'role': 'assistant',
   'content': 'Please note that ```git``` has two configuration commands:\n```git config filter..clean\ngit config filter..smudge\n```\nThe ```clean``` filter can be used e.g. to sort all records in a CSV file except the top row, thus making re-ordering irrelevant.\nQuoting ```gitattributes(5)``` :\n\nA filter driver consists of a clean command and a smudge command, either of which can be left unspecified. Upon checkout, when the smudge command is specified, the command is fed the blob object from its standard input, and its standard output is used to update the worktree file. Similarly, the clean command is used to convert the c

In [ ]:
greeting_and_qa_data = json.load(open("/content/drive/MyDrive/llm_from_scratch/datasets/greeting_and_qa.json"))

In [ ]:
print(len(greeting_and_qa_data),  "\n")
greeting_and_qa_data[44]

452 



{'messages': [{'role': 'user',
   'content': "What's the background of your developer?"},
  {'role': 'assistant',
   'content': 'My developer, Muhammad Toqeer Yasir, has strong background in artificial intelligence, machine learning, deep learning, and natural language processing. He created me from the ground up in 2025 as a comprehensive AI project.'}]}

In [ ]:
# with open('/content/drive/MyDrive/llm_from_scratch/datasets/instruction_data_alpaca_gpt4.json', 'r') as f:
#   alpaca_data = json.load(f)
# print(len(alpaca_data),  "\n")
# alpaca_data[44]

In [ ]:
with open('/content/drive/MyDrive/llm_from_scratch/datasets/orca_agentinstruct_10k_balanced.json', 'r') as f:
    orca_instruction_data = json.load(f)

for item in orca_instruction_data:
    if isinstance(item.get('messages'), str):
        item['messages'] = json.loads(item['messages'])

orca_instruction_data[0]

{'messages': [{'role': 'system', 'content': ''},
  {'role': 'user',
   'content': "Create an outline for a presentation that includes five engaging and informative slides to test the audience's understanding of the concepts discussed in your talk, such as p-value functions, compatibility intervals, and the misconceptions about p-values. Each slide should present a multiple-choice question related to these topics. Prepare talking points that explain the correct answers and why they are correct, ensuring the explanations are clear and suitable for an educational presentation."},
  {'role': 'assistant',
   'content': 'Title: Understanding P-Values and Statistical Inference\n\nSlide 1: Introduction to P-Values\n- Multiple-Choice Question: What does a p-value indicate in hypothesis testing?\n  A) The probability that the null hypothesis is true\n  B) The probability of observing the data, or something more extreme, if the null hypothesis is true\n  C) The probability that the alternative hy

In [ ]:
random.seed(44)
orca_instruction_data = random.sample(orca_instruction_data, 1518)
len(orca_instruction_data)

1518

## **Combining datasets to create a single one and shuffle it.**

In [ ]:
merged_data = greeting_and_qa_data + orca_instruction_data + instruction_data

In [ ]:
random.seed(44)
random.shuffle(merged_data)

In [ ]:
print(len(merged_data))
merged_data[44]

3000


{'messages': [{'role': 'user',
   'content': 'write a story about the grinch as if he was a lovecraftian monster'},
  {'role': 'assistant',
   'content': 'An Excerpt from "The Krumpet Horror":\nThere are no words to properly express the horror of that day...\nI had been investigating a strange case of delluminating Christmas lights when I came across a strange clue: green hair and a lingering odor, like sulfur.\nThis led me down a dark path of research that no Whovian should ever tred. For I uncovered knowledge of a creature so foul, so terrible, that one could go mad at the very thought...\nThe fool I was. I followed the clues up the mountain. Up, to the dark, shattered peak of Mt. Krumpet. And there, set into the cold stone, I found the entrance to His haunted lair.\nI crept inside, slowly. The dim lights from further down reflected off the damp cave walls, allowing me sufficient light to proceed.\nFrom somewhere below me, I heard a cackling of laughter. It echoed around me, seeping 

In [ ]:
class ConvertDataIntoTokenIds(torch.utils.data.Dataset):
    def __init__(self, data, max_length=512):
        self.data = data
        self.tokenizer = tiktoken.get_encoding('gpt2')
        self.max_length = max_length
        self.eot_token = 50256

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        if "instruction" in item:
            text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"

        elif item["messages"][0]["role"] == "system":
            user_msg = item["messages"][1]["content"]
            assistant_msg = item["messages"][2]["content"]

        else:
            user_msg = item["messages"][0]["content"]
            assistant_msg = item["messages"][1]["content"]

        text = f"User: {user_msg}\nAssistant: {assistant_msg}"

        tokens = self.tokenizer.encode(text)

        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        if len(tokens) < self.max_length:
            padding = [self.eot_token] * (self.max_length - len(tokens))
            tokens = tokens + padding

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'labels': torch.tensor(tokens, dtype=torch.long)
        }

In [ ]:
# creating dataset
train_dataset = ConvertDataIntoTokenIds(merged_data)

In [ ]:
train_dataset[244]

{'input_ids': tensor([12982,    25, 13610,   281,  2708, 11142,   262, 22588, 13433,   286,
         17103,   338,  9238,   290, 23669,  1080,    11,   351,   257,  2962,
           319,  3912, 12336,   290, 15148,  1626,  2872,  5101,    13,   383,
          2708,   815,   307, 20793,   351,  2438,  6096,   326, 19418,  2219,
         45716,   618,  7219,   351, 22546,  1366,   287,  2872,  6299,    13,
           383,  3597,  3918,   815,   307,  6276,   290, 30304,    11,  8998,
           379, 19898, 17103, 24867,   508,   389,  5385,   351,   262,  3303,
           338,  4096, 15582,   475,   743,  6531,   351,   663, 23669,  3173,
            13,   383,  8216,   815,   307,  9856,   290,  4622,  8766,    11,
           355,   611,   340,   547,   257,  6276,  5698,   393, 10314,    13,
           198,   198,    16,    13,  7253,   262,  2708,   351,   281,  9793,
           284, 17103,   338,  9238,  1080,    11, 36360,   262,  6817,   286,
          4547, 23669,   290,  9238,  3

In [ ]:
# setting monkey patch for forward method:
class GPTConfig(PretrainedConfig):
    def __init__(self, **kwargs):
        # config
        self.vocab_size = 50257
        self.context_length = 1024
        self.emb_dim = 1280
        self.n_heads = 20
        self.n_layers = 36
        self.drop_rate = 0.1
        self.qkv_bias = True
        super().__init__(**kwargs)

# attaching config to model
model.config = GPTConfig()

In [ ]:
#  attaching monkey patch for forword function:
def hf_forward(self, input_ids=None, labels=None, attention_mask=None, **kwargs):
    batch_size, seq_len = input_ids.shape

    with torch.amp.autocast('cuda', enabled=True):
        tok_embeds = self.tok_emb(input_ids)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=input_ids.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

    loss = None
    if labels is not None:
        loss_fct = torch.nn.CrossEntropyLoss()
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

    return (loss, logits) if loss is not None else logits

model.forward = hf_forward.__get__(model, type(model))

In [ ]:
model.unload()
print('Model unloaded successfully.')

Model unloaded successfully.


In [ ]:
def prepare_inputs_for_generation(self, input_ids, **kwargs):
    return {"input_ids": input_ids, **kwargs}

if not hasattr(model, 'prepare_inputs_for_generation'):
    model.prepare_inputs_for_generation = prepare_inputs_for_generation.__get__(model, type(model))

In [ ]:
# applying lora addapters for reducing model size:
def setup_lora_model(model):

    target_modules = [
        "W_query", "W_key", "W_value",
        "out_proj",
        "out_head"
    ]

    # LoRA configuration
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # applying LoRA
    model = get_peft_model(model, lora_config)

    return model

# applying LoRA to model
model = setup_lora_model(model)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# training arguments for peft:
import transformers
transformers.logging.set_verbosity_info()
training_args = TrainingArguments(
    # output & saving
    output_dir="/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model",
    save_strategy="no",

    # training hyperparameters
    learning_rate=2e-4,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    weight_decay=0.01,

    # optimization
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # precision & performance
    fp16=True,
    dataloader_pin_memory=False,
    dataloader_drop_last=True,
    remove_unused_columns=False,

    # logging & monitoring
    logging_strategy="steps",
    logging_steps=10,
    disable_tqdm=False,
    report_to="none",
)

PyTorch: setting up devices


In [ ]:
torch.cuda.empty_cache()
gc.collect()

7

In [ ]:
# creating trianer:
os.environ["WANDB_DISABLED"] = "true"
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

Using auto half precision backend


In [ ]:
# clearing memory:
torch.cuda.empty_cache()
gc.collect()

print("Started training...")
trainer.train()
print("Training completed!")

Started training...


***** Running training *****
  Num examples = 452
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 57
  Number of trainable parameters = 6,722,832


Step,Training Loss
10,62.404500
20,13.535900
30,4.098100
40,2.972300
50,2.834600




Training completed. Do not forget to share your model on huggingface.co/models =)




Training completed!


In [ ]:
# def test_model(model, input_text="", max_new_tokens=256):
#     encoding = tiktoken.get_encoding('gpt2')

#     # Tokenize once
#     input_ids = encoding.encode(input_text)
#     input_tensor = torch.tensor([input_ids]).to(device)

#     model.eval()
#     with torch.no_grad():
#         # Use model.generate - much more efficient
#         outputs = model.generate(
#             input_tensor,
#             max_new_tokens=max_new_tokens,
#             temperature=0.7,
#             do_sample=True,
#             pad_token_id=encoding.eot_token,
#             eos_token_id=encoding.eot_token,
#             early_stopping=True
#         )

#     # Decode and clean
#     response = encoding.decode(outputs[0].tolist())
#     response = response.replace(input_text, '').replace('<|endoftext|>', '').strip()

#     return response

# text = input("Ask something to AI.")
# print("Assistant: ", test_model(text))

🧪 Testing the fine-tuned model:



KeyboardInterrupt: 

In [ ]:
# # After instruction fine-tuning, merge LoRA into base model
# merged_model = model.merge_and_unload()

# # Save the MERGED weights (now base model has instruction knowledge)
# torch.save(merged_model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/finetuned_model.pth")

# print("✅ Saved unified model with instruction knowledge")